In [1]:
import torch
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader



import sys, os
sys.path.append(os.path.abspath(os.path.join(os.getcwd(),'..' ,'..')))
from Data.class_dataset import MRIDataset
from Model.model import build_vit3d

# --- Cargar datos de test ---

df=pd.read_csv("ext_test_data.csv")

test_imgs = df["Path"].tolist()
test_ages = df["Age"].tolist()
test_dataset = MRIDataset(test_imgs, test_ages)
test_loader = DataLoader(test_dataset, batch_size=8, shuffle=False)

# --- Cargar modelo ---
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = build_vit3d()
state_dict = torch.load("../../Training/Trained_models/model_9.pth", map_location=device)
# Si las claves tienen 'module.' al inicio, elimínalo
if any(k.startswith('module.') for k in state_dict.keys()):
    from collections import OrderedDict
    new_state_dict = OrderedDict()
    for k, v in state_dict.items():
        new_key = k.replace('module.', '', 1)
        new_state_dict[new_key] = v
    state_dict = new_state_dict
model.load_state_dict(state_dict)
model.to(device)
model.eval()



ViT3D(
  (patch_to_embedding): Linear(in_features=4096, out_features=1024, bias=True)
  (dropout): Dropout(p=0.1, inplace=False)
  (transformer): Transformer(
    (layers): ModuleList(
      (0-5): 6 x ModuleList(
        (0): Residual(
          (fn): PreNorm(
            (norm): LayerNorm((1024,), eps=1e-05, elementwise_affine=True)
            (fn): Attention(
              (to_qkv): Linear(in_features=1024, out_features=1536, bias=False)
              (to_out): Sequential(
                (0): Linear(in_features=512, out_features=1024, bias=True)
                (1): Dropout(p=0.1, inplace=False)
              )
            )
          )
        )
        (1): Residual(
          (fn): PreNorm(
            (norm): LayerNorm((1024,), eps=1e-05, elementwise_affine=True)
            (fn): FeedForward(
              (net): Sequential(
                (0): Linear(in_features=1024, out_features=2048, bias=True)
                (1): GELU(approximate='none')
                (2): Dropout(p=

In [2]:
# --- Evaluar ---
all_preds = []
all_ages = []
with torch.no_grad():
    for imgs, ages in test_loader:
        imgs = imgs.to(device)
        preds = model(imgs)
        all_preds.append(preds.cpu())
        all_ages.append(ages.unsqueeze(1).cpu())
all_preds = torch.cat(all_preds).numpy().flatten()
all_ages = torch.cat(all_ages).numpy().flatten()
all_paths = df["Path"].tolist()

mae = mean_absolute_error(all_ages, all_preds)
r2 = r2_score(all_ages, all_preds)
print(f"Test MAE: {mae:.2f}")
print(f"Test R2: {r2:.2f}")

Test MAE: 6.52
Test R2: 0.79


In [3]:
#crear un df con las edades reales, predichas y el ID
results_df = pd.DataFrame({
    "ID": all_paths,
    "Age": all_ages,
    "Prediction": all_preds,
    "Error": all_preds - all_ages, 
    'Absolute Error': np.abs(all_preds - all_ages)
})
results_df['ID']=results_df['ID'].replace('/data/lautaro/quasiraw/', '', regex=True)
results_df['ID']=results_df['ID'].replace('.nii.gz', '', regex=True)
results_df.sort_values(by='Absolute Error', ascending=True, inplace=True)
results_df.to_csv("test_predictions_6.csv", index=False)

In [4]:
results_df

,ID,Age,Prediction,Error,Absolute Error
345,/data/lautaro/quasiraw_ext/RRIB_sub-016,47.0,47.015881,0.015881,0.015881
139,/data/lautaro/quasiraw_ext/CP0175,56.0,55.979980,-0.020020,0.020020
1377,/data/lautaro/quasiraw_ext/131_S_6170_I1526394,61.0,60.976433,-0.023567,0.023567
633,/data/lautaro/quasiraw_ext/013_S_6780_I1221673,63.0,62.974213,-0.025787,0.025787
833,/data/lautaro/quasiraw_ext/027_S_6183_I959453,66.0,66.029518,0.029518,0.029518
...,...,...,...,...,...
757,/data/lautaro/quasiraw_ext/021_S_6987_I1475785,66.0,36.269917,-29.730083,29.730083
1531,/data/lautaro/quasiraw_ext/177_S_6335_I1327456,69.0,38.071461,-30.928539,30.928539
1105,/data/lautaro/quasiraw_ext/082_S_6564_I1049352,71.0,39.707752,-31.292248,31.292248
463,/data/lautaro/quasiraw_ext/RRIB_sub-141,25.0,61.113598,36.113598,36.113598
